In [1]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content
!rm -rf verbascribe-exam
!git clone https://github.com/muhammadwaqarsaleem/verbascribe-exam.git
%cd /content/verbascribe-exam
!unzip -q -o /content/drive/MyDrive/verbascribe/audio_train.zip -d /content/verbascribe-exam/
!pip install -q -U torchao
!pip install -q peft accelerate jiwer soundfile pydantic pydub pandas

Mounted at /content/drive
/content
Cloning into 'verbascribe-exam'...
remote: Enumerating objects: 81, done.
remote: Counting objects: 100% (81/81), done.
remote: Compressing objects: 100% (56/56), done.
remote: Total 81 (delta 42), reused 62 (delta 23), pack-reused 0 (from 0)
Receiving objects: 100% (81/81), 86.04 KiB | 7.17 MiB/s, done.
Resolving deltas: 100% (42/42), done.
/content/verbascribe-exam


In [2]:
import csv, random, numpy as np, soundfile as sf, torch
from dataclasses import dataclass
from transformers import (WhisperForConditionalGeneration, WhisperProcessor,
                          Seq2SeqTrainer, Seq2SeqTrainingArguments)
from peft import LoraConfig, get_peft_model

MODEL_ID = "distil-whisper/distil-small.en"
processor = WhisperProcessor.from_pretrained(MODEL_ID)
model = WhisperForConditionalGeneration.from_pretrained(MODEL_ID)
model.generation_config.forced_decoder_ids = None
model.config.use_cache = False

model = get_peft_model(model, LoraConfig(r=32, lora_alpha=64, lora_dropout=0.05,
                                         bias="none", target_modules=["q_proj", "v_proj"]))
model.print_trainable_parameters()

rows = list(csv.DictReader(open("data/train_manifest.csv", encoding="utf-8")))
texts = sorted({r["text"] for r in rows})
random.seed(42); random.shuffle(texts)
val_texts = set(texts[:8])
train_rows = [r for r in rows if r["text"] not in val_texts]
val_rows   = [r for r in rows if r["text"] in val_texts]
print("train clips:", len(train_rows), "| val clips:", len(val_rows))

SOT = processor.tokenizer.convert_tokens_to_ids("<|startoftranscript|>")

def prep(r):
    a, sr = sf.read(r["path"])
    feats = processor.feature_extractor(a.astype(np.float32), sampling_rate=16000).input_features[0]
    labels = processor.tokenizer(" " + r["text"]).input_ids
    if labels[0] == SOT:          # Trainer already prepends the start token
        labels = labels[1:]
    return {"input_features": feats, "labels": labels}

train_ds = [prep(r) for r in train_rows]
val_ds   = [prep(r) for r in val_rows]

@dataclass
class Collator:
    def __call__(self, batch):
        feats = torch.tensor(np.stack([b["input_features"] for b in batch]))
        lab = [b["labels"] for b in batch]
        m = max(len(l) for l in lab)
        labels = torch.full((len(lab), m), -100)
        for i, l in enumerate(lab):
            labels[i, :len(l)] = torch.tensor(l)
        return {"input_features": feats, "labels": labels}

args = Seq2SeqTrainingArguments(
    output_dir="models/whisper_lora_ckpt", per_device_train_batch_size=8,
    gradient_accumulation_steps=2, learning_rate=1e-4, num_train_epochs=5,
    warmup_steps=20, fp16=True, eval_strategy="epoch", save_strategy="no",
    logging_steps=10, report_to="none", remove_unused_columns=False, label_names=["labels"])

trainer = Seq2SeqTrainer(model=model, args=args, train_dataset=train_ds,
                         eval_dataset=val_ds, data_collator=Collator())
trainer.train()
model.save_pretrained("models/whisper_lora_adapter")
print("Adapter saved.")

# backup to Drive (v2)
!mkdir -p /content/drive/MyDrive/verbascribe/whisper_lora_adapter_v2
!cp -r models/whisper_lora_adapter/* /content/drive/MyDrive/verbascribe/whisper_lora_adapter_v2/

W1113 12:47:26.948000 1022 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1113 12:47:27.068000 1022 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


preprocessor_config.json:   0%|          | 0.00/339 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/2.26k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/282k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/999k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.41M [00:00<?, ?B/s]

... (7 outputs left out)

generation_config.json:   0%|          | 0.00/1.92k [00:00<?, ?B/s]

trainable params: 1,966,080 || all params: 168,098,304 || trainable%: 1.1696
train clips: 216 | val clips: 24


Epoch,Training Loss,Validation Loss
1,1.497672,0.464940
2,0.977547,0.248182
3,0.412262,0.151223
4,0.219470,0.131847
5,0.162510,0.129751


Adapter saved.


In [4]:
%cd /content/verbascribe-exam
import os, re, csv, gc, numpy as np, pandas as pd, jiwer, torch, transformers
transformers.logging.set_verbosity_error()
from pydub import AudioSegment
from sklearn.metrics import roc_auc_score
from src.asr_engine import ASREngine

# Cell 2 ke training objects GPU se hatao
for name in ["model", "trainer"]:
    if name in globals():
        del globals()[name]
gc.collect(); torch.cuda.empty_cache()

# adapter Drive se lao
!mkdir -p models
!cp -r /content/drive/MyDrive/verbascribe/whisper_lora_adapter_v2 models/

AUDIO_DIR = "/content/drive/MyDrive/verbascribe/test_recordings"
OUT16 = "/content/drive/MyDrive/verbascribe/test_recordings_16k"
os.makedirs(OUT16, exist_ok=True)

ref_by_id = {int(r["id"]): r["text"] for r in
             csv.DictReader(open("data/academic_vocab_prompts.csv", encoding="utf-8"))
             if r["split"] == "test"}

def norm(t):
    return " ".join(re.sub(r"[^a-z0-9\s]", " ", t.lower()).split())

# ---- 1. mp3 -> 16 kHz mono ----
clips = {}
for f in sorted(os.listdir(AUDIO_DIR)):
    m = re.fullmatch(r"(\d+)\.(mp3|m4a|wav)", f)
    if not m: continue
    i = int(m.group(1))
    seg = AudioSegment.from_file(os.path.join(AUDIO_DIR, f)).set_frame_rate(16000).set_channels(1)
    seg.export(os.path.join(OUT16, f"{i}.wav"), format="wav")
    x = np.array(seg.get_array_of_samples()).astype(np.float32)
    clips[i] = x / float(1 << (8 * seg.sample_width - 1))
durs = [len(v) / 16000 for v in clips.values()]
print("clips:", len(clips), "| duration s: min %.1f mean %.1f max %.1f" % (min(durs), np.mean(durs), max(durs)))

# ---- 2. transcribe with both models (geo + min confidence) ----
def transcribe_all(adapter):
    eng = ASREngine(adapter_path=adapter)
    out = {}
    for i in sorted(clips):
        eng.conf_mode = "geo"; g = eng.transcribe(clips[i], sr=16000)
        eng.conf_mode = "min"; mn = eng.transcribe(clips[i], sr=16000)
        out[i] = (g, mn)
    del eng; gc.collect(); torch.cuda.empty_cache()
    return out

results = {"base": transcribe_all(None),
           "lora": transcribe_all("models/whisper_lora_adapter_v2")}

# ---- 3. keywords (rare academic terms) ----
KEYWORDS = set("""centripetal torque fermion pauli boson doppler inductor electromagnetic gauss flux
endoplasmic reticulum peptide polymerase nucleotides phospholipid bilayer osmosis semipermeable solute
photosynthesis chlorophyll electrochemical missense nonsense quicksort tokenization dijkstra asymmetric
encryption laplace algebraic orthogonal determinant invertible isomers hybridization tetrahedral methane
exothermic enthalpy titration reagent chromatography electronegativity catalysts fission nucleus genome
differentiation""".split())

# ---- 4. metrics + per-word labels ----
clip_rows, word_rows, skipped = [], [], 0
for model_name, res in results.items():
    for i, (g, mn) in res.items():
        ref, hyp = norm(ref_by_id[i]), norm(g.raw_transcript)
        wo = jiwer.process_words(ref, hyp)
        ref_ok = [False] * len(ref.split()); hyp_ok = [False] * len(hyp.split())
        for ch in wo.alignments[0]:
            if ch.type == "equal":
                for k in range(ch.ref_start_idx, ch.ref_end_idx): ref_ok[k] = True
                for k in range(ch.hyp_start_idx, ch.hyp_end_idx): hyp_ok[k] = True
        kw_idx = [k for k, w in enumerate(ref.split()) if w in KEYWORDS]
        clip_rows.append(dict(clip_id=i, model=model_name, ref=ref, hyp=hyp,
                              errors=wo.substitutions + wo.deletions + wo.insertions,
                              n_ref=len(ref.split()), kw_total=len(kw_idx),
                              kw_miss=sum(not ref_ok[k] for k in kw_idx)))
        toks, cg, cm = [], [], []
        for a, b in zip(g.word_confidences, mn.word_confidences):
            for t in norm(a.word).split():
                toks.append(t); cg.append(a.confidence); cm.append(b.confidence)
        if toks != hyp.split():
            skipped += 1; continue
        for k, t in enumerate(toks):
            word_rows.append(dict(clip_id=i, model=model_name, word=t,
                                  conf_geo=round(cg[k], 4), conf_min=round(cm[k], 4),
                                  correct=int(hyp_ok[k])))

cdf, wdf = pd.DataFrame(clip_rows), pd.DataFrame(word_rows)
cdf.to_csv("data/test_results.csv", index=False)
wdf.to_csv("data/word_confidences_test.csv", index=False)
!cp data/test_results.csv data/word_confidences_test.csv /content/drive/MyDrive/verbascribe/

# ---- 5. summary ----
print("\n=== RESULTS (40 real recordings) ===")
for m in ["base", "lora"]:
    d = cdf[cdf.model == m]
    print(f"{m:5s} WER: {d.errors.sum()/d.n_ref.sum():.4f} | "
          f"keyword error rate: {d.kw_miss.sum()}/{d.kw_total.sum()} = {d.kw_miss.sum()/d.kw_total.sum():.4f}")

b = cdf[cdf.model == "base"].sort_values("clip_id"); l = cdf[cdf.model == "lora"].sort_values("clip_id")
eb, el, n = b.errors.values, l.errors.values, b.n_ref.values
rng = np.random.default_rng(42)
diffs = []
for _ in range(10000):
    idx = rng.integers(0, len(n), len(n))
    diffs.append((el[idx].sum() - eb[idx].sum()) / n[idx].sum())
lo, hi = np.percentile(diffs, [2.5, 97.5])
print(f"\nWER difference (LoRA - base): {(el.sum()-eb.sum())/n.sum():+.4f}  95% bootstrap CI [{lo:+.4f}, {hi:+.4f}]")

print("\n=== Confidence AUROC (word correct vs confidence) ===")
for m in ["base", "lora"]:
    d = wdf[wdf.model == m]
    print(m, "| words:", len(d), "| wrong:", int((d.correct == 0).sum()),
          "| AUROC geo: %.3f | min: %.3f" % (roc_auc_score(d.correct, d.conf_geo),
                                              roc_auc_score(d.correct, d.conf_min)))
print("clips skipped (token mismatch):", skipped)

/content/verbascribe-exam


/usr/local/lib/python3.13/dist-packages/pydub/utils.py:300: SyntaxWarning: invalid escape sequence '\('
  m = re.match('([su]([0-9]{1,2})p?) \(([0-9]{1,2}) bit\)$', token)
/usr/local/lib/python3.13/dist-packages/pydub/utils.py:301: SyntaxWarning: invalid escape sequence '\('
  m2 = re.match('([su]([0-9]{1,2})p?)( \(default\))?$', token)
/usr/local/lib/python3.13/dist-packages/pydub/utils.py:310: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(flt)p?( \(default\))?$', token):
/usr/local/lib/python3.13/dist-packages/pydub/utils.py:314: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(dbl)p?( \(default\))?$', token):


clips: 40 | duration s: min 3.9 mean 5.6 max 9.2


Loading weights:   0%|          | 0/287 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/287 [00:00<?, ?it/s]


=== RESULTS (40 real recordings) ===
base  WER: 0.1114 | keyword error rate: 14/51 = 0.2745
lora  WER: 0.0977 | keyword error rate: 13/51 = 0.2549

WER difference (LoRA - base): -0.0136  95% bootstrap CI [-0.0356, +0.0091]

=== Confidence AUROC (word correct vs confidence) ===
base | words: 444 | wrong: 44 | AUROC geo: 0.869 | min: 0.878
lora | words: 441 | wrong: 38 | AUROC geo: 0.885 | min: 0.893
clips skipped (token mismatch): 0


In [5]:
!cp data/test_results.csv data/word_confidences_test.csv /content/drive/MyDrive/verbascribe/
from google.colab import files
files.download("data/word_confidences_test.csv")
files.download("data/test_results.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>